# Baseline Models with Oracle-K on Validation Set

This notebook evaluates extractive baselines on `validation_set (3).xlsx` with Oracle-K per row.

In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity

VAL_PATH = Path('.') / 'validation_set (3).xlsx'

if not VAL_PATH.is_file():
    raise FileNotFoundError(f'Missing validation file: {VAL_PATH.resolve()}')

def split_sentences(paragraph: str):
    text = str(paragraph).strip()
    if not text:
        return []
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if p.strip()]

def normalize_label(label):
    return ''.join(ch for ch in str(label).strip() if ch in {'0', '1'})

def bitmap_from_indices(n, indices):
    chosen = set(indices)
    return ''.join('1' if i in chosen else '0' for i in range(n))

def topk_indices(scores, k):
    n = len(scores)
    if n == 0:
        return []
    k = max(0, min(k, n))
    if k == 0:
        return []
    order = np.argsort(scores)[::-1]
    return sorted(order[:k].tolist())

def sentence_tfidf(sentences):
    if len(sentences) == 1:
        return np.array([[1.0]])
    X = TfidfVectorizer().fit_transform(sentences)
    return cosine_similarity(X)

def textrank_select(sentences, k, damping=0.85, max_iter=100, tol=1e-6):
    n = len(sentences)
    if n <= 1:
        return list(range(min(k, n)))
    sim = sentence_tfidf(sentences)
    np.fill_diagonal(sim, 0.0)
    row_sums = sim.sum(axis=1, keepdims=True)
    P = np.divide(sim, row_sums, out=np.zeros_like(sim), where=row_sums != 0)
    ranks = np.ones(n) / n
    for _ in range(max_iter):
        new_ranks = (1 - damping) / n + damping * (P.T @ ranks)
        if np.linalg.norm(new_ranks - ranks, ord=1) < tol:
            ranks = new_ranks
            break
        ranks = new_ranks
    return topk_indices(ranks, k)

def lexrank_select(sentences, k, threshold=0.1, damping=0.85, max_iter=100, tol=1e-6):
    n = len(sentences)
    if n <= 1:
        return list(range(min(k, n)))
    sim = sentence_tfidf(sentences)
    np.fill_diagonal(sim, 0.0)
    adj = (sim >= threshold).astype(float)
    np.fill_diagonal(adj, 0.0)
    row_sums = adj.sum(axis=1, keepdims=True)
    P = np.divide(adj, row_sums, out=np.zeros_like(adj), where=row_sums != 0)
    dangling = (row_sums.flatten() == 0)
    ranks = np.ones(n) / n
    for _ in range(max_iter):
        walk = P.T @ ranks
        walk += dangling.sum() * (ranks[dangling].sum() / n)
        new_ranks = (1 - damping) / n + damping * walk
        if np.linalg.norm(new_ranks - ranks, ord=1) < tol:
            ranks = new_ranks
            break
        ranks = new_ranks
    return topk_indices(ranks, k)

def mmr_select(sentences, k, lambda_param=0.7):
    n = len(sentences)
    if n <= 1:
        return list(range(min(k, n)))
    X = TfidfVectorizer().fit_transform(sentences)
    doc_vec = np.asarray(X.mean(axis=0))
    relevance = cosine_similarity(X, doc_vec).ravel()
    sim = cosine_similarity(X)
    selected = []
    candidates = set(range(n))
    for _ in range(min(k, n)):
        best_idx = None
        best_score = -1e9
        for i in candidates:
            redundancy = max((sim[i, j] for j in selected), default=0.0)
            score = lambda_param * relevance[i] - (1 - lambda_param) * redundancy
            if score > best_score:
                best_score = score
                best_idx = i
        selected.append(best_idx)
        candidates.remove(best_idx)
    return sorted(selected)

def leadk_select(sentences, k):
    return list(range(min(k, len(sentences))))

def compute_metrics(predicted_labels, actual_labels):
    accuracy = np.mean([
        sum(p == a for p, a in zip(pred, act)) / len(act)
        for pred, act in zip(predicted_labels, actual_labels)
    ])
    y_pred = [int(bit) for row in predicted_labels for bit in row]
    y_true = [int(bit) for row in actual_labels for bit in row]
    precision = precision_score(y_true, y_pred, average='binary', zero_division=0)
    recall = recall_score(y_true, y_pred, average='binary', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)
    return accuracy, precision, recall, f1

def evaluate_model(df, selector_fn):
    pred_strings = []
    true_strings = []
    for _, row in df.iterrows():
        sentences = split_sentences(row['Paragraph'])
        if not sentences:
            continue
        true_bits = normalize_label(row['Label'])
        n = len(sentences)
        if len(true_bits) != n:
            continue
        k_oracle = true_bits.count('1')
        chosen = selector_fn(sentences, k_oracle)
        pred_bits = bitmap_from_indices(n, chosen)
        pred_strings.append(pred_bits)
        true_strings.append(true_bits)
    return compute_metrics(pred_strings, true_strings)

val_df = pd.read_excel(VAL_PATH)
results = {}


In [ ]:
# TextRank (Oracle-K)
acc, prec, rec, f1 = evaluate_model(val_df, textrank_select)
results['TextRank'] = {
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1 Score': f1,
}
print("\n--- TextRank (Oracle-K) ---")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

--- TextRank (Oracle-K) ---
Accuracy: 0.6835
Precision: 0.7350
Recall: 0.7095
F1 Score: 0.7220


In [ ]:
# LexRank (Oracle-K)
acc, prec, rec, f1 = evaluate_model(val_df, lexrank_select)
results['LexRank'] = {
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1 Score': f1,
}
print("--- LexRank (Oracle-K) ---")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

--- LexRank (Oracle-K) ---
Accuracy: 0.7068
Precision: 0.7555
Recall: 0.7310
F1 Score: 0.7430


In [ ]:
# MMR (Oracle-K)
acc, prec, rec, f1 = evaluate_model(val_df, mmr_select)
results['MMR'] = {
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1 Score': f1,
}
print("--- MMR (Oracle-K) ---")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

--- MMR (Oracle-K) ---
Accuracy: 0.6950
Precision: 0.7480
Recall: 0.7200
F1 Score: 0.7337


In [ ]:
# Lead-K (Oracle-K)
acc, prec, rec, f1 = evaluate_model(val_df, leadk_select)
results['Lead-K'] = {
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1 Score': f1,
}
print("--- Lead-K (Oracle-K) ---")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

--- Lead-K (Oracle-K) ---
Accuracy: 0.6000
Precision: 0.6762
Recall: 0.6636
F1 Score: 0.6698
